In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import pyquist as pq
import icm_plotly
from icm_plotly import RED, GOLD, STEEL, IRON

Drag the slider: each step adds the next harmonic of the sawtooth
recipe — harmonic $k$ at amplitude $1/k$ — to a running sum. On the left,
the partial sum (Carnegie red) bends closer to the ideal sawtooth (steel
gray), with the newest harmonic drawn in gold; on the right, each bar of
the recipe lights up as its harmonic joins.

In [ ]:
# autorun
f0 = 220.0                            # fundamental (A3)
n_max = 12
T = 2 / f0                            # two cycles on screen
t = np.linspace(0.0, T, 900, endpoint=False)

# The sawtooth recipe: harmonic k at amplitude 1/k (2/pi normalizes the
# full series to peak +-1). Row n of `sums` is the wave after 1..n+1.
k = np.arange(1, n_max + 1)
partials = (2 / np.pi) * np.sin(2 * np.pi * f0 * k[:, None] * t[None, :]) / k[:, None]
sums = np.cumsum(partials, axis=0)
ideal = 1.0 - 2.0 * ((f0 * t) % 1.0)  # what the infinite series converges to

def figure():
    fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
                        horizontal_spacing=0.12)

    # left: the ideal sawtooth as a fixed reference, the sum drawn on top
    fig.add_scatter(x=t * 1000, y=ideal, mode="lines",
                    line=dict(color=STEEL, width=1.6), row=1, col=1)
    fig.add_scatter(x=t * 1000, y=sums[0], mode="lines",
                    line=dict(color=RED, width=2), row=1, col=1)
    fig.add_scatter(x=t * 1000, y=partials[0], mode="lines",
                    line=dict(color=GOLD, width=1.4), row=1, col=1)
    fig.update_xaxes(title_text="Time (ms)", fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.4, 1.4], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)

    # right: the recipe, 1/k per harmonic; joined bars light up
    fig.add_bar(x=k, y=1 / k, marker_color=[GOLD] + [STEEL] * (n_max - 1),
                row=1, col=2)
    fig.update_xaxes(title_text="Harmonic k", fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[0, 1.05], title_text="Amplitude 1/k",
                     fixedrange=True, row=1, col=2)
    return fig

def controls(fig):
    n = widgets.IntSlider(description="Harmonics", min=1, max=n_max, value=1)

    # the defaults snapshot the arrays — the page's notebooks share one kernel
    def update(n, sums=sums, partials=partials, n_max=n_max):
        with fig.batch_update():
            fig.data[1].y = sums[n - 1]
            fig.data[2].y = partials[n - 1]
            # joined harmonics in red, the newest in gold, the rest waiting
            fig.data[3].marker.color = (
                [RED] * (n - 1) + [GOLD] + [STEEL] * (n_max - n))

    widgets.interactive_output(update, {"n": n})
    return widgets.VBox([n])

icm_plotly.show(figure, controls)

**Now hear it.** The same recipe rendered as sound: half a second each
of 1, 2, 4, 8, then 16 harmonics of the same fundamental. Listen for the
tone sharpening toward a buzz as the upper harmonics arrive.

In [ ]:
sr = 44100
f0 = 220.0
seg = np.arange(int(0.5 * sr)) / sr   # half a second per step
steps = [1, 2, 4, 8, 16]

clips = []
for n in steps:
    kk = np.arange(1, n + 1)
    saw = (np.sin(2 * np.pi * f0 * kk[:, None] * seg) / kk[:, None]).sum(axis=0)
    saw *= 0.5 / np.abs(saw).max()    # consistent, comfortable loudness
    saw[:220] *= np.linspace(0, 1, 220)   # 5 ms fades: no clicks between steps
    saw[-220:] *= np.linspace(1, 0, 220)
    clips.append(saw)

pq.play(pq.Audio(np.concatenate(clips), sample_rate=sr))